# 01 — EDA i kontrakt danych (Sprint 1)

Cel notebooka: poznać dane wybranego datasetu i podjąć decyzje, które trafią do kodu:

- **input normalization** — reguły w `normalize_input()` (`src/asi_project/data.py`),
- **training-only filtering** — reguły w `filter_training_rows()` (jeśli są potrzebne),
- **typy cech** — listy `features.numeric` i `features.categorical` w `project.yaml`,
- **metryka główna** — uzasadnienie w karcie projektu w README.

Jak pracować:

- uruchamiajcie notebook od góry (Restart & Run All); po zmianie `project.yaml` uruchomcie
  ponownie komórkę ustawień,
- komórki oznaczone **TODO** uzupełnia zespół; odpowiedzi tekstowe wpisujcie w miejsce
  `*(odpowiedź)*`,
- nie zmieniajcie kontraktu prowadzącej (`src/asi_project/datasets.yaml`).

## 0. Ustawienia

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

from asi_project.config import get_contract, load_project_config
from asi_project.data import (
    encode_target,
    feature_list_problems,
    filter_training_rows,
    is_text_column,
    load_dataset,
    normalize_input,
    target_classes,
    validate,
)

config = load_project_config()
contract = get_contract(config)
DATA_MODE = os.environ.get("ASI_DATA_MODE", "full")

print(f"Dataset: {contract.title} ({contract.dataset_id})")
print(f"Target: {contract.target} | klasa pozytywna: {contract.positive_class}")
print(f"Moment predykcji: {contract.prediction_moment}")

## 1. Wczytanie danych

Pełne dane pobiera `python scripts/download_data.py`. Nazwy kolumn są ujednolicane przy
wczytywaniu (białe znaki), wartości — jeszcze nie.

In [ ]:
df_raw = load_dataset(config, contract, mode=DATA_MODE)
print(df_raw.shape)
df_raw.head()

## 2. Inspekcja: typy, braki, liczba unikalnych wartości

In [ ]:
overview = pd.DataFrame(
    {
        "dtype": df_raw.dtypes.astype(str),
        "braki": df_raw.isna().sum(),
        "unikalne": df_raw.nunique(),
    },
)
overview

**TODO — obserwacje (2–4 zdania):** które kolumny mają nieoczekiwany typ? Gdzie są braki?
Które kolumny liczbowe są w rzeczywistości kategoriami?

Kolumny OperatingSystems, Browser, Region, TrafficType, a także PageValues mają nieoczekiwany typ. W zbiorze nie ma braków. Kolumny liczbowe, które są w rzeczywistości kategoriami to OperatingSystems, Browser, Region i TrafficType.

## 3. Kontrakt danych

Kontrakt prowadzącej określa target, klasę pozytywną, ALLOWED FEATURES (jedyne kolumny
dozwolone w X) i EXCLUDED FEATURES wraz z powodem wykluczenia.

In [ ]:
problems = validate(df_raw, contract)
print("Kontrakt danych:", "OK" if not problems else problems)
print("ALLOWED FEATURES:", list(contract.allowed_features))
print("EXCLUDED FEATURES:")
for column, reason in contract.excluded_features.items():
    status = "jest w danych" if column in df_raw.columns else "brak w pliku"
    print(f"- {column} ({status}): {reason}")

## 4. Target i klasa pozytywna

In [ ]:
print("Klasy targetu:", target_classes(df_raw[contract.target]))
y = encode_target(df_raw[contract.target].dropna(), contract.positive_class)
print(f"Udział klasy pozytywnej: {y.mean():.3f}")
ax = y.value_counts().sort_index().plot(kind="bar", title="Target (1 = klasa pozytywna)")
ax.set_xlabel("klasa")
plt.show()

**TODO — metryka główna (2–3 zdania):** którą metrykę (accuracy, precision, recall, F1)
wybieracie jako główną i dlaczego, biorąc pod uwagę udział klasy pozytywnej i koszt błędów
w waszym problemie? To samo uzasadnienie wpiszcie do karty projektu w README.

Jako metrykę główną wybieramy F1. Przez nie zbalansowanie klasy accuracy nie jest dobrą metryką do oceny, a F1 zarówno uwzględni procent osób wskazanych przez model jako osoby dokonujące zakupu które naprawdę dokonało zakupu, jak i procent klientów którzy naprawdę dokonali zakupu i zostali wskazani przez model.

## 5. EDA

**TODO:** co najmniej dwie analizy — jedna dla cechy numerycznej, jedna dla kategorycznej.
Wybierajcie wyłącznie spośród ALLOWED FEATURES.

In [ ]:
# TODO (S1): rozkład wybranej cechy numerycznej w podziale na klasy targetu, np.
# df_raw.groupby(contract.target)["<cecha>"].describe()
def plot_numeric_by_target(data, column, target):

    _, ax = plt.subplots(figsize=(8, 4))
    for label, group in data.groupby(target):
        ax.hist(group[column], bins=30, alpha=0.5, density=True, label=f"{target} = {label}")
    ax.set_title(f"Rozkład {column} w podziale na {target}")
    ax.set_xlabel(column)
    ax.set_ylabel("gęstość (osobno dla każdej klasy)")
    ax.legend()
    plt.show()
    print("Mediana w każdej klasie:")
    print(data.groupby(target)[column].median().to_string())

In [ ]:
plot_numeric_by_target(df_raw, "Administrative", contract.target)

In [ ]:
df_raw.groupby(contract.target)["Administrative"].describe()

In [ ]:
plot_numeric_by_target(df_raw, "Administrative_Duration", contract.target)

In [ ]:
df_raw.groupby(contract.target)["Administrative_Duration"].describe()

In [ ]:
plot_numeric_by_target(df_raw, "Informational", contract.target)

In [ ]:
df_raw.groupby(contract.target)["Informational"].describe()

In [ ]:
plot_numeric_by_target(df_raw, "Informational_Duration", contract.target)

In [ ]:
df_raw.groupby(contract.target)["Informational_Duration"].describe()

In [ ]:
plot_numeric_by_target(df_raw, "ProductRelated", contract.target)

In [ ]:
df_raw.groupby(contract.target)["ProductRelated"].describe()

In [ ]:
plot_numeric_by_target(df_raw, "ProductRelated_Duration", contract.target)

In [ ]:
df_raw.groupby(contract.target)["ProductRelated_Duration"].describe()

In [ ]:
plot_numeric_by_target(df_raw, "BounceRates", contract.target)

In [ ]:
df_raw.groupby(contract.target)["BounceRates"].describe()

In [ ]:
plot_numeric_by_target(df_raw, "ExitRates", contract.target)

In [ ]:
df_raw.groupby(contract.target)["ExitRates"].describe()

In [ ]:
plot_numeric_by_target(df_raw, "SpecialDay", contract.target)

In [ ]:
df_raw.groupby(contract.target)["SpecialDay"].describe()

In [ ]:
# TODO (S1): udział klasy pozytywnej w kategoriach wybranej cechy kategorycznej, np.
# pd.crosstab(df_raw["<cecha>"], df_raw[contract.target], normalize="index")
def plot_category_by_target(data, column, target, positive_label="Yes", max_categories=20):
    n_categories = data[column].nunique()
    if n_categories > max_categories:
        raise ValueError(f"Kolumna {column!r} ma {n_categories} różnych wartości.")
    is_positive = data[target] == positive_label
    rate_label = f"odsetek {target} = {positive_label}"
    summary = pd.DataFrame(
        {
            "liczba wierszy": data.groupby(column).size(),
            rate_label: is_positive.groupby(data[column]).mean().round(3),
        }
    ).sort_values(rate_label)
    overall_rate = is_positive.mean()
    _, ax = plt.subplots(figsize=(8, 1.5 + 0.5 * n_categories))
    ax.barh(summary.index.astype(str), summary[rate_label])
    ax.axvline(overall_rate, color="gray", linestyle="--", label=f"cały zbiór: {overall_rate:.1%}")
    ax.set_title(f"{rate_label} w kategoriach kolumny {column}")
    ax.set_xlabel(rate_label)
    ax.legend()
    plt.show()
    return summary.sort_values(rate_label, ascending=False)

In [ ]:
plot_category_by_target(df_raw, "Month", contract.target, positive_label=True)

In [ ]:
plot_category_by_target(df_raw, "OperatingSystems", contract.target, positive_label=True)

In [ ]:
plot_category_by_target(df_raw, "Browser", contract.target, positive_label=True)

In [ ]:
plot_category_by_target(df_raw, "Region", contract.target, positive_label=True)

In [ ]:
plot_category_by_target(df_raw, "TrafficType", contract.target, positive_label=True)

In [ ]:
plot_category_by_target(df_raw, "VisitorType", contract.target, positive_label=True)

In [ ]:
plot_category_by_target(df_raw, "Weekend", contract.target, positive_label=True)

**TODO — wnioski z EDA (3–5 zdań):**

Klienci dokonujący zakupu zazwyczaj odwiedzają więcej stron i spędzają na nich więcej czasu.
Klienci dokonujący zakupu typowo mają zauważalnie niższy BounceRates i ExitRates w porównaniu do klientów nie kupujących.
Nowi klienci mają w uśrednieniu największą szansę na dokonanie zakupu ze wszystkich klientów.
Typowo jest większa szansa, że klient dokona zakupu w okresie przed świętami bożego narodzenia (wrzesień - listopad).

## 6. Input normalization

Input normalization działa na pojedynczym rekordzie, nie używa targetu ani statystyk zbioru
i jest idempotentna. Typowe kandydatki: białe znaki, puste napisy, oczywiste reprezentacje
braków, ujednolicenie zapisu wartości. Uzupełnianie braków medianą **nie jest** input
normalization — wymaga `fit`, więc należy do model preprocessing.

In [ ]:
text_columns = [column for column in df_raw.columns if is_text_column(df_raw[column])]
for column in text_columns:
    values = df_raw[column].dropna().astype(str)
    padded = int((values != values.str.strip()).sum())
    empty = int((values.str.strip() == "").sum())
    top = values.value_counts().head(6).to_dict()
    print(f"{column}: spacje na brzegach={padded}, puste={empty}, najczęstsze={top}")

**TODO — reguły input normalization:** jakie reguły dopisujecie do `normalize_input()` i dlaczego
są bezpieczne przy predykcji pojedynczego rekordu? Po zmianie funkcji uruchomcie `pytest`.

Zmienne nie zawierają znaków białych i mają spójne wartości. Dodatkową regułą będzie zmiana typów kolumn OperatingSystems, Browser, Region i TrafficType, których typy nie są poprawne i należy je zmienić z liczbowych (int64) na kategoryczny (category). Konwersja jest bezpieczna przy predykcji pojedynczego rekordu, ponieważ nie korzysta z targetu oraz nie wymaga uczenia parametrów czy obliczania na podstawie całego zbioru.

In [ ]:
normalized = normalize_input(df_raw)
changed = (normalized.astype(str) != df_raw.astype(str)).sum()
changed[changed > 0]

## 7. Training-only filtering

Training-only filtering działa tylko przy przygotowaniu danych treningowych i może jedynie
usuwać rekordy. Domyślnie `filter_training_rows()` usuwa rekordy bez targetu.

In [ ]:
print("Rekordy bez targetu:", int(df_raw[contract.target].isna().sum()))
print("Dokładne duplikaty wierszy:", int(df_raw.duplicated().sum()))
filtered = filter_training_rows(normalize_input(df_raw), contract)
print(f"Po training-only filtering: {len(filtered)} z {len(df_raw)} rekordów")

**TODO — decyzja o training-only filtering:** czy usuwacie dodatkowo jakieś rekordy
(np. dokładne duplikaty)? Dlaczego tak albo dlaczego nie?

W naszych danych jeden rekord oznacza sesję zakupową, nie koniecznie unikalne doświadczenie klienta. Nie jesteśmy w stanie określić czy dany wiersz jest sesją zapisaną dwa razy, czy może to dwie różne sesje. Z tego powodu pozostawiamy dokładne duplikaty wierszy, ponieważ mogą one wnosić wartość predykcyjną dla modelu.

## 8. Typy cech → `project.yaml`

**TODO:** wpiszcie do `project.yaml` listy `features.numeric` i `features.categorical`
(wyłącznie ALLOWED FEATURES), zapiszcie plik i uruchomcie ponownie komórkę ustawień oraz
komórkę poniżej.

In [ ]:
print("Numeryczne:", list(config.numeric_features))
print("Kategoryczne:", list(config.categorical_features))
print("Problemy z deklaracją:", feature_list_problems(contract, config.model_features) or "brak")
print("Kontrakt z deklaracją:", validate(normalize_input(df_raw), contract, config) or "OK")

## 9. Podsumowanie decyzji

**TODO — w 3–5 punktach:** decyzje o input normalization, training-only filtering, typach cech
i metryce głównej. Te decyzje będą punktem wyjścia do ADR-1 w Sprincie 2.

1. **Metryka główna:**  
Wybraną przez nas główną metryką jest f1-score dla targetu Revenue i klasy pozytywnej (True=1). Klasa pozytywna stanowi 15.5% obserwacji, accuracy byłoby mylące w ocenie jakości modelu, dlatego wybraliśmy F1.

2. **Typy cech i input normalization:**  
W danych nie ma braków, pustych znaków, kategorie są spójne tak jak i zakresy wartości. Zmieniono typ kolumn 'OperatingSystems', 'Browser', 'Region' i 'TrafficType' z int64 na category.

3. **Training-only filtering:**  
Każdy rekord to sesja zakupowa i chociaż mogą być identyczne i tak wnoszą wartość predykcyjną (częstość danego zakupu z tym samym targetem jest pomocna), nie usuwaliśmy duplikatów rekordów. W danych nie wystąpiły rekordy bez targetu.
